In [1]:
%load_ext autoreload
%autoreload 2

from database.manager import DatabaseManager
from analysis.seasonality import Seasonality
from analysis.plotting import SeasonalityPlotter
from analysis.feature_creation import FeatureCreator
from Mini_Tools.export_to_excel import export_to_excel
import pandas as pd
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from analysis.trend_score import get_stability_metrics

db = DatabaseManager()
sznlty = Seasonality(db)
plotter = SeasonalityPlotter()
fc = FeatureCreator()

In [ ]:
result = sznlty.working_day_expression_seasonality("CO", "J27 - 3*M27 + 3*Q27 - V27 ", start_year=2014, end_year=2027, window_days=350)
plotter.plot_seasonality(result, title="seasonality")
export_to_excel(result, filename="seasonality_report_final.xlsx")

In [2]:
# Call the function
metrics = get_stability_metrics(
    sznlty=sznlty, 
    symbol="CO", 
    expression="J27 - 3*M27 + 3*Q27 - V27", 
    start_year=2014, 
    end_year=2027, 
    window_days=350,
    threshold=0.1  # Optional: defaults are already set in the function
)

# 1. Print the Final Scores
print(f"Latest Year Score: {metrics['latest_score']}")
print(f"General Trend Score: {metrics['trend_score']}")

Latest Year Score: 80.0
General Trend Score: 88.7


In [ ]:
def plot_technical_with_cutoff(result, window_bb=20, window_rsi=14, cutoff=43):
    # 1. Filter data to exclude everything after the cutoff
    df_full = result['combined'].sort_index()
    df = df_full[df_full.index <= -cutoff] 
    
    year_cols = [c for c in df.columns if str(c).isdigit() and len(str(c)) == 4]
    latest_year = max(year_cols)
    hist_years = [c for c in year_cols if c != latest_year]
    
    # 2. Extract series and DROP NaNs
    full_index = df.index
    latest_series = df[latest_year].dropna()
    hist_mean = df[hist_years].mean(axis=1)
    
    # 3. Indicators
    sma = latest_series.rolling(window=window_bb).mean()
    std = latest_series.rolling(window=window_bb).std()
    
    # Option C: Trailing Z-Score
    z_score = (latest_series - sma) / std
    
    # RSI Calculation
    delta = latest_series.diff()
    gain = delta.clip(lower=0).rolling(window_rsi).mean()
    loss = (-delta.clip(upper=0)).rolling(window_rsi).mean()
    rs = gain / loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    rsi.loc[(loss == 0) & (gain > 0)] = 100

    # 4. Plotting (3 Subplots)
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 14), sharex=True, 
                                        gridspec_kw={'height_ratios': [3, 1, 1]})
    
    # --- Chart 1: Price, Historical Mean, SMA, and BB ---
    ax1.plot(full_index, hist_mean, label='Historical Seasonal Mean', color='blue', linestyle=':', lw=1.5)
    ax1.plot(latest_series.index, latest_series, label=f'Current Price ({latest_year})', color='black', lw=2)
    ax1.plot(sma.index, sma, label=f'{window_bb}d Rolling Mean', color='orange', lw=1, alpha=0.8)
    ax1.fill_between(latest_series.index, sma - (std * 2), sma + (std * 2), color='gray', alpha=0.1, label='Bollinger Bands')
    ax1.set_title(f"Lifecycle Analysis & Entry Signals (Cutoff: {cutoff}d)")
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.2)

    # --- Chart 2: Trailing Z-Score (Option C) ---
    ax2.plot(z_score.index, z_score, color='red', label='Trailing Z-Score')
    ax2.axhline(2, color='black', linestyle='--', alpha=0.3)
    ax2.axhline(-2, color='black', linestyle='--', alpha=0.3)
    ax2.axhline(0, color='gray', lw=1)
    ax2.set_ylabel("Z-Score")
    ax2.set_ylim(-4, 4)
    ax2.grid(True, alpha=0.2)
    ax2.legend(loc='upper left')

    # --- Chart 3: RSI ---
    ax3.plot(rsi.index, rsi, color='purple', label='RSI')
    ax3.axhline(70, color='red', linestyle='--', alpha=0.3)
    ax3.axhline(30, color='green', linestyle='--', alpha=0.3)
    ax3.set_ylim(0, 100)
    ax3.set_ylabel("RSI")
    ax3.grid(True, alpha=0.2)
    
    plt.xlabel("Days to Expiry")
    plt.tight_layout()
    plt.show()
    return pd.DataFrame({
    "z_score": z_score,
    "rsi": rsi,
    })

# Execution
plot_technical_with_cutoff(metrics['raw_result'])

In [5]:
# Z-score signals confirmed by RSI extreme zones
upper_rsi = 60
lower_rsi = 30

upper_rsi_zone = df["rsi"].between(upper_rsi, 100, inclusive="both")
lower_rsi_zone = df["rsi"].between(0, lower_rsi, inclusive="both")

upper_signal = (
    ((df["z_score"] >= 2.0) & upper_rsi_zone)
    | ((df["z_score"] >= 1.9) & upper_rsi_zone)
)
lower_signal = (
    ((df["z_score"] <= -2.0) & lower_rsi_zone)
    | ((df["z_score"] <= -1.9) & lower_rsi_zone)
)

# -1 = upper/sell signal, +1 = lower/buy signal, 0 = no signal
df["signal"] = np.select(
    [upper_signal, lower_signal],
    [-1, 1],
    default=0,
)

df[["z_score", "rsi", "signal"]].tail()

NameError: name 'df' is not defined